In [1]:
import numpy as np;
import pandas as pd;
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,Dropout

In [3]:
text_data = pd.read_csv("dataset.csv")
text_data.head()

,text,humor
0,"Joe biden rules out 2020 bid: 'guys, i'm not r...",False
1,Watch: darvish gave hitter whiplash with slow ...,False
2,What do you call a turtle without its shell? d...,True
3,5 reasons the 2016 election feels so personal,False
4,"Pasco police shot mexican migrant from behind,...",False


In [4]:
text_data = text_data.drop('humor', axis=1)
text_data.head()

,text
0,"Joe biden rules out 2020 bid: 'guys, i'm not r..."
1,Watch: darvish gave hitter whiplash with slow ...
2,What do you call a turtle without its shell? d...
3,5 reasons the 2016 election feels so personal
4,"Pasco police shot mexican migrant from behind,..."


In [5]:
def clean_text(text):
    # 1. Convert to string and lowercase
    text = str(text).lower()
    # 2. Replace everything that ISN'T a letter or space with a space
    # (This keeps the words but removes punctuation/numbers)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 3. Remove extra whitespace
    text = " ".join(text.split())
    return text

# Apply this to your column
text_data['text'] = text_data['text'].apply(clean_text)
text_data.head()


,text
0,joe biden rules out bid guys i m not running
1,watch darvish gave hitter whiplash with slow p...
2,what do you call a turtle without its shell dead
3,reasons the election feels so personal
4,pasco police shot mexican migrant from behind ...


In [19]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
# builds dictionary from data
tokenizer.fit_on_texts(text_data['text'])

input_sequences = []

for line in text_data['text']:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Now pad these and split into X and y
# Change 'post' to 'pre'
padded_sequences = pad_sequences(input_sequences, maxlen=20, padding='pre')

# Post-padding + Filter = No Data: In post style, the "next word" is almost always a 0. Your filter deletes those rows, leaving you with almost no data to train on.
# Forgetfulness: LSTMs read left-to-right. With post, the model reads words, then a bunch of zeros, then tries to guess. It "forgets" the words by the time it reaches the guess.
# GPU Crashes: The fast GPU driver (cuDNN) is mathematically incompatible with right-side masking, leading to the Assertion Failed error you saw.

X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

# Create X and y as usual
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

# FIND rows where y is NOT 0
mask = y != 0
X = X[mask]
y = y[mask]




In [20]:
vocab_size = len(tokenizer.word_index) + 1
# 👉 Using recurrent_dropout disables fast cuDNN and
#  forces a slower, flexible LSTM that supports
#  masking—removing the GPU error but reducing
#  training speed.
model = Sequential([
    # Remove mask_zero to keep the GPU happy and fast
    Embedding(10001, 128, input_length=19),

    # Stacked LSTMs for "Sense"
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(128),
    Dropout(0.2),

    Dense(10001, activation='softmax')
])



In [21]:
# sparse_categorical_crossentropy is a memory-saver that lets you use integers (1, 5, 59853) as labels instead
# of massive one-hot matrices (mostly zeros) that would crash your RAM.

model.compile(
    loss='sparse_categorical_crossentropy', # Use this to save memory!
    optimizer='adam',
    metrics=['accuracy']
)

In [22]:
model.fit(X, y, epochs=20, batch_size=128)


Epoch 1/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 97s 13ms/step - accuracy: 0.1250 - loss: 6.2335
Epoch 2/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 94s 13ms/step - accuracy: 0.1669 - loss: 5.6367
Epoch 3/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 95s 13ms/step - accuracy: 0.1813 - loss: 5.4092
Epoch 4/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 95s 13ms/step - accuracy: 0.1908 - loss: 5.2564
Epoch 5/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 144s 13ms/step - accuracy: 0.1978 - loss: 5.1436
Epoch 6/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 98s 13ms/step - accuracy: 0.2040 - loss: 5.0552
Epoch 7/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 140s 13ms/step - accuracy: 0.2087 - loss: 4.9826
Epoch 8/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 92s 12ms/step - accuracy: 0.2133 - loss: 4.9202
Epoch 9/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 92s 12ms/step - accuracy: 0.2167 - loss: 4.8681
Epoch 10/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 142s 12ms/step - accuracy: 0.2202 - loss: 4.8200
Epoch 11/20
7467/7467 ━━━━━━━━━━━━━━━━━━━━ 94s 13ms/step - accuracy: 0.2233 - loss: 4.7777
Epoch

In [24]:
import numpy as np

def generate_text(seed_text, next_words=10):
    for _ in range(next_words):
        # 1. Tokenize and pad
        token_list = tokenizer.texts_to_sequences([seed_text])
        token_list = pad_sequences(token_list, maxlen=19, padding='pre')

        # 2. Get probabilities
        predicted_probs = model.predict(token_list, verbose=0)[0]

        # 3. Pick from the TOP 3 most likely words (adds "sense" and variety)
        top_indices = np.argsort(predicted_probs)[-3:]
        predicted_index = np.random.choice(top_indices)

        # 4. Skip OOV/Padding
        if predicted_index <= 1:
            predicted_index = np.argsort(predicted_probs)[-4]

        # 5. Find the word in dictionary
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        if not output_word: break
        seed_text += " " + output_word

    return seed_text

In [34]:
# Try giving it a very simple setup to see if it stays on track:
generate_text("why did the student")
generate_text("success is")
# print(generate_text("the best way to"))
# print(generate_text("knowledge is"))


'success is a lot i can t even see my wife in'